# ⚙️ Nexora — Feature Engineering Pipeline
**Phase 5: Customer, Product & Interaction Features**

### Overview & Data Leakage Prevention
In this phase, we compute multidimensional predictive features while strictly adhering to temporal boundaries:
1. **Customer Behavioral Features**: RFM attributes, order interval variance, basket size velocity, department diversity.
2. **Product Catalog Features**: Historical demand volume, global reorder rate, average cart position.
3. **User-Product Interaction Features**: Reorder frequency ratio, order lag since last purchase, cart priority.
4. **Zero Leakage**: Features are calculated strictly from prior orders, predicting outcomes in the subsequent train order.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="deep")
DATA_DIR = Path("../data/processed")

df_cust = pd.read_parquet(DATA_DIR / "customer_features.parquet")
df_prod = pd.read_parquet(DATA_DIR / "product_features.parquet")
df_up = pd.read_parquet(DATA_DIR / "user_product_features.parquet")

print(f"Customer Feature Matrix:     {len(df_cust):,} rows x {df_cust.shape[1]} features")
print(f"Product Feature Matrix:      {len(df_prod):,} rows x {df_prod.shape[1]} features")
print(f"User-Product Feature Matrix: {len(df_up):,} rows x {df_up.shape[1]} features")
df_cust.head()

---
## 1. Customer Feature Correlation Matrix
Examining interdependencies between total orders, interval cadence, basket size, and department diversity.

In [ ]:
cols = ["user_total_orders", "user_avg_order_interval", "user_avg_basket_size", "user_reorder_rate", "user_unique_departments"]
plt.figure(figsize=(9, 6))
sns.heatmap(df_cust[cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title("Customer Behavioral Features Correlation Matrix", fontsize=13, fontweight="bold")
plt.show()

---
## 2. User-Product Interaction Features & Target Balance
Analyzing historical user-product order rate vs. future purchase outcome in the subsequent order.

In [ ]:
print("Target Class Distribution (Reordered in Next Order):")
print(df_up["target"].value_counts(normalize=True))

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_up.sample(10000, random_state=42), x="target", y="up_order_rate", palette="Set1")
plt.title("User Product Order Rate vs. Target (Next Order Reorder)", fontsize=13, fontweight="bold")
plt.xlabel("Target (0 = Not Reordered, 1 = Reordered)")
plt.ylabel("Historical User-Product Order Rate")
plt.show()